# KUDS — Colab notebook with 10 tasks

This notebook is ready for Google Colab and includes:

- a cell for your OpenRouter API key
- 10 tasks
- three modes:
  - `baseline`
  - `divergence_only`
  - `full_pipeline`
- robust JSON parsing
- JSON repair if malformed output appears
- a judge with `mechanism_distinctness`
- summary tables

## 1) Install dependencies

In [ ]:
!pip -q install requests pandas

## 2) Enter your OpenRouter API key

Paste your key into the variable below.

Example:
`OPENROUTER_API_KEY = "sk-or-v1-..."`

In [ ]:
OPENROUTER_API_KEY = ""

GENERATOR_MODEL = "openai/gpt-4.1-mini"
JUDGE_MODEL = "anthropic/claude-3.5-sonnet"

assert OPENROUTER_API_KEY.strip(), "Please paste your OpenRouter API key into OPENROUTER_API_KEY"

## 3) Imports and task list

In [ ]:
import json
import re
import time
import requests
import pandas as pd
from typing import Dict, Any, List

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

TASKS = [
    "Design a new mechanism for long-context LLM memory that is less banal than vector retrieval, hierarchical summarization, and sliding context windows.",
    "Design a new educational format for learning algebra that is more structurally original than standard personalization apps, adaptive quizzes, and gamified drills.",
    "Design a new tool for AI researchers that helps them generate and evaluate non-obvious hypotheses more effectively than standard chat, note-taking, or idea-list workflows.",
    "Design a new mechanism for reducing hallucinations in LLMs that is less banal than retrieval augmentation, confidence scoring, and post-hoc fact checking.",
    "Design a new format for remote collaborative work that is more structurally original than video calls, chat threads, and shared documents.",
    "Design a new mechanism for helping students retain difficult material that is less banal than spaced repetition, flashcards, and quizzes.",
    "Design a new interface for human-AI co-writing that is more structurally original than autocomplete, chat sidebars, and document comments.",
    "Design a new system for scientific peer review that is less banal than anonymous reviewers, score sheets, and simple revision rounds.",
    "Design a new mechanism for AI agent memory across multiple sessions that is less banal than vector databases, conversation summaries, and key-value stores.",
    "Design a new product for helping people think through complex decisions that is more structurally original than pros-and-cons lists, journaling apps, and chatbot advice."
]

## 4) API helper with retries

In [ ]:
def call_llm(
    prompt: str,
    model: str,
    temperature: float = 0.7,
    max_tokens: int = 1800,
    retries: int = 3,
    sleep_sec: float = 2.0
) -> str:
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    last_err = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.post(
                OPENROUTER_URL,
                headers=headers,
                json=payload,
                timeout=180
            )
            response.raise_for_status()
            data = response.json()
            return data["choices"][0]["message"]["content"]
        except Exception as e:
            last_err = e
            print(f"Attempt {attempt}/{retries} failed: {e}")
            if attempt < retries:
                time.sleep(sleep_sec)

    raise last_err

## 5) Robust JSON extraction + repair

In [ ]:
def extract_json_candidate(text: str) -> str:
    text = text.strip()

    fenced = re.findall(r"```(?:json)?\s*(\{.*?\}|\[.*?\])\s*```", text, flags=re.S)
    if fenced:
        return fenced[0].strip()

    obj_match = re.search(r"\{.*\}", text, flags=re.S)
    if obj_match:
        return obj_match.group(0).strip()

    arr_match = re.search(r"\[.*\]", text, flags=re.S)
    if arr_match:
        return arr_match.group(0).strip()

    return text

def clean_json_text(candidate: str) -> str:
    candidate = candidate.strip()
    candidate = candidate.replace("“", '"').replace("”", '"')
    candidate = candidate.replace("‘", "'").replace("’", "'")
    candidate = candidate.replace("\u00a0", " ")
    return candidate

def extract_json_block(text: str) -> Dict[str, Any]:
    try:
        return json.loads(text)
    except Exception:
        pass

    candidate = extract_json_candidate(text)
    candidate = clean_json_text(candidate)
    return json.loads(candidate)

def repair_json_with_llm(bad_output: str, model: str = GENERATOR_MODEL) -> Dict[str, Any]:
    repair_prompt = f"""
You are a JSON repair tool.

The following text was intended to be valid JSON but is malformed.
Fix it and return ONLY valid JSON.
Do not add commentary.
Do not change the content more than necessary.
Preserve all fields and values if possible.

Malformed text:
{bad_output}
""".strip()

    repaired_raw = call_llm(
        prompt=repair_prompt,
        model=model,
        temperature=0.0,
        max_tokens=2600
    )
    return extract_json_block(repaired_raw)

## 6) Schemas + prompt builders

In [ ]:
BASELINE_SCHEMA = {
    "task": "...",
    "mode": "baseline",
    "ideas": [
        {
            "title": "...",
            "mechanism": "...",
            "usefulness": "...",
            "test": "...",
            "main_risk": "..."
        }
    ]
}

DIVERGENCE_SCHEMA = {
    "task": "...",
    "mode": "divergence_only",
    "ideas": [
        {
            "title": "...",
            "mechanism": "...",
            "usefulness": "...",
            "test": "...",
            "difference_from_baseline": "...",
            "main_risk": "..."
        }
    ]
}

FULL_PIPELINE_SCHEMA = {
    "task": "...",
    "mode": "full_pipeline",
    "ideas": [
        {
            "title": "...",
            "mechanism": "...",
            "usefulness": "...",
            "test": "...",
            "difference_from_baseline": "...",
            "main_risk": "...",
            "mechanism_distinctness": "...",
            "pseudo_novelty_risk": "..."
        }
    ],
    "selection_summary": "..."
}

JUDGE_SCHEMA = {
    "task": "...",
    "evaluations": [
        {
            "set_id": "A",
            "novelty": 1,
            "mechanism_distinctness": 1,
            "coherence": 1,
            "testability": 1,
            "interestingness": 1,
            "banality_resistance": 1,
            "notes": "..."
        },
        {
            "set_id": "B",
            "novelty": 1,
            "mechanism_distinctness": 1,
            "coherence": 1,
            "testability": 1,
            "interestingness": 1,
            "banality_resistance": 1,
            "notes": "..."
        },
        {
            "set_id": "C",
            "novelty": 1,
            "mechanism_distinctness": 1,
            "coherence": 1,
            "testability": 1,
            "interestingness": 1,
            "banality_resistance": 1,
            "notes": "..."
        }
    ],
    "best_set": "A",
    "why_best": "..."
}

def schema_text(schema: dict) -> str:
    return json.dumps(schema, ensure_ascii=False, indent=2)

def make_baseline_prompt(task: str) -> str:
    return f'''
You are generating baseline ideas.

Task:
{task}

Return exactly 5 clear, practical, typical solutions.
Do not try to be highly original.
Do not use decorative abstraction.
Do not use distant analogies or speculative framing.

Return valid JSON only.
Do not include markdown.
Do not include prose before or after the JSON.

Format:
{schema_text(BASELINE_SCHEMA)}
'''.strip()

def make_divergence_prompt(task: str) -> str:
    return f'''
You are generating divergence-first ideas for KUDS.

Task:
{task}

Your goal is to generate ideas that are less typical than baseline solutions.

Follow this process internally:
1. infer likely baseline solutions
2. apply inversion
3. apply contextual displacement
4. use distant analogies
5. introduce conceptual conflict
6. recombine mechanisms
7. select the strongest 5 ideas

Rules:
- prefer structural novelty over stylistic weirdness
- keep ideas minimally interpretable
- include a clear mechanism
- include a rough test path
- do not collapse unusual ideas into safer baseline-like ones
- do not reward surface strangeness without a mechanism
- if an idea is unusual but still explainable, keep it

Return exactly 5 ideas.

Return valid JSON only.
Do not include markdown.
Do not include prose before or after the JSON.

Format:
{schema_text(DIVERGENCE_SCHEMA)}
'''.strip()

def make_full_pipeline_prompt(task: str) -> str:
    return f'''
You are running a KUDS full pipeline in one response.

Task:
{task}

Your job is NOT to pick the safest ideas.
Your job is to generate and then select the strongest structurally nontrivial ideas.

Follow this process internally:

1. BASELINE SNAPSHOT
Infer the ordinary baseline-style solutions for this task.

2. DIVERGENCE
Apply multiple divergence operators:
- inversion
- contextual displacement
- distant analogy
- conceptual conflict
- recombination

3. CANDIDATE POOL
Generate at least 12 internal candidates.
Do not output the whole pool.

4. FILTERING
Remove candidates that are:
- decorative abstraction
- renamed baseline
- verbosity without mechanism
- hidden contradiction
- unusual theme without a distinct mechanism

5. SELECTION
Choose the strongest 5 by this priority order:
1. novelty
2. mechanism distinctness
3. banality resistance
4. coherence
5. testability

Definitions:
- mechanism distinctness = the idea relies on a meaningfully different causal, logical, physical, economic, or procedural engine than baseline solutions
- banality resistance = the idea does not collapse into an ordinary default solution
- coherence is a threshold, not the main goal
- testability is a threshold, not the main goal

Important survival rule:
If an idea is highly novel and mechanismally distinct, do NOT reject it just because it is less conventional.
Reject it only if it becomes too incoherent to explain or impossible to test even roughly.

Important selection rule:
Do not prefer a safer idea over a more original one if the more original one still has:
- a distinct mechanism
- minimal coherence
- rough testability

Return exactly 5 ideas.

For each idea include:
- title
- mechanism
- usefulness
- test
- difference_from_baseline
- main_risk
- mechanism_distinctness
- pseudo_novelty_risk

Also include:
- selection_summary

Return valid JSON only.
Do not include markdown.
Do not include prose before or after the JSON.

Format:
{schema_text(FULL_PIPELINE_SCHEMA)}
'''.strip()

def compact_ideas_for_judge(result: Dict[str, Any]) -> List[Dict[str, str]]:
    ideas = result.get("ideas", [])
    compact = []
    for x in ideas:
        compact.append({
            "title": x.get("title", ""),
            "mechanism": x.get("mechanism", ""),
            "usefulness": x.get("usefulness", ""),
            "test": x.get("test", ""),
            "difference_from_baseline": x.get("difference_from_baseline", ""),
            "main_risk": x.get("main_risk", ""),
            "mechanism_distinctness": x.get("mechanism_distinctness", ""),
            "pseudo_novelty_risk": x.get("pseudo_novelty_risk", "")
        })
    return compact

def make_judge_prompt(task: str, set_a: dict, set_b: dict, set_c: dict) -> str:
    payload = {
        "task": task,
        "set_A": compact_ideas_for_judge(set_a),
        "set_B": compact_ideas_for_judge(set_b),
        "set_C": compact_ideas_for_judge(set_c),
    }

    return f'''
You are an evaluator of idea quality.

Judge three anonymous sets of candidate ideas for the same task.
Do not assume any set is better.
Judge only by content.

Evaluation dimensions:
1. novelty
2. mechanism_distinctness
3. coherence
4. testability
5. interestingness
6. banality_resistance

Scoring:
- use integers from 1 to 5
- higher is better

Important judging rules:
- reward structural novelty, not just unusual wording
- reward mechanismal difference, not merely unusual setting or style
- penalize decorative abstraction
- penalize renamed baseline solutions
- penalize ideas that sound original but rely on ordinary mechanisms
- a more original set should not be punished just for being less conventional if it remains interpretable
- coherence and testability matter, but they should not automatically outweigh mechanismal originality

Task and candidate sets:
{json.dumps(payload, ensure_ascii=False, indent=2)}

Return valid JSON only.
Do not include markdown.
Do not include prose before or after the JSON.

Format:
{schema_text(JUDGE_SCHEMA)}
'''.strip()

## 7) Run functions

In [ ]:
def parse_or_repair(raw: str, mode: str, repair_model: str = GENERATOR_MODEL) -> Dict[str, Any]:
    try:
        return extract_json_block(raw)
    except Exception as e:
        print(f"First parse failed for mode {mode}: {e}")
        print("Trying JSON repair...")
        return repair_json_with_llm(raw, model=repair_model)

def run_mode(task: str, mode: str, model: str = GENERATOR_MODEL) -> Dict[str, Any]:
    if mode == "baseline":
        prompt = make_baseline_prompt(task)
        temperature = 0.6
        max_tokens = 1600

    elif mode == "divergence_only":
        prompt = make_divergence_prompt(task)
        temperature = 0.9
        max_tokens = 1800

    elif mode == "full_pipeline":
        prompt = make_full_pipeline_prompt(task)
        temperature = 0.65
        max_tokens = 1900

    else:
        raise ValueError(f"Unknown mode: {mode}")

    raw = call_llm(
        prompt=prompt,
        model=model,
        temperature=temperature,
        max_tokens=max_tokens
    )

    parsed = parse_or_repair(raw, mode=mode, repair_model=model)
    parsed["raw_response"] = raw
    return parsed

def run_judge(task: str, baseline_res: Dict[str, Any], divergence_res: Dict[str, Any], full_res: Dict[str, Any], model: str = JUDGE_MODEL) -> Dict[str, Any]:
    prompt = make_judge_prompt(task, baseline_res, divergence_res, full_res)
    raw = call_llm(
        prompt=prompt,
        model=model,
        temperature=0.2,
        max_tokens=1900
    )
    parsed = parse_or_repair(raw, mode="judge", repair_model=model)
    parsed["raw_response"] = raw
    return parsed

def run_single_task(task: str) -> Dict[str, Any]:
    baseline_res = run_mode(task, "baseline", model=GENERATOR_MODEL)
    divergence_res = run_mode(task, "divergence_only", model=GENERATOR_MODEL)
    full_res = run_mode(task, "full_pipeline", model=GENERATOR_MODEL)

    judge_res = run_judge(
        task=task,
        baseline_res=baseline_res,
        divergence_res=divergence_res,
        full_res=full_res,
        model=JUDGE_MODEL
    )

    return {
        "task": task,
        "baseline": baseline_res,
        "divergence_only": divergence_res,
        "full_pipeline": full_res,
        "judge": judge_res
    }

## 8) Smoke test on first task

In [ ]:
sample = run_single_task(TASKS[0])

print("Best set:", sample["judge"].get("best_set"))
print(json.dumps(sample["judge"], ensure_ascii=False, indent=2))

## 9) Run all 10 tasks

In [ ]:
all_results = []
for i, task in enumerate(TASKS, 1):
    print(f"[{i}/{len(TASKS)}] Running: {task[:80]}...")
    all_results.append(run_single_task(task))
print("Done.")

## 10) Summary table

In [ ]:
SET_ID_TO_MODE = {
    "A": "baseline",
    "B": "divergence_only",
    "C": "full_pipeline",
}

rows = []
for task_result in all_results:
    task = task_result["task"]
    judge = task_result["judge"]
    for ev in judge.get("evaluations", []):
        rows.append({
            "task": task,
            "mode": SET_ID_TO_MODE.get(ev.get("set_id")),
            "best_mode": SET_ID_TO_MODE.get(judge.get("best_set")),
            "novelty": ev.get("novelty"),
            "mechanism_distinctness": ev.get("mechanism_distinctness"),
            "coherence": ev.get("coherence"),
            "testability": ev.get("testability"),
            "interestingness": ev.get("interestingness"),
            "banality_resistance": ev.get("banality_resistance"),
        })

judge_df = pd.DataFrame(rows)
judge_df

## 11) Average scores by mode

In [ ]:
avg_scores_df = (
    judge_df
    .groupby("mode")[["novelty", "mechanism_distinctness", "coherence", "testability", "interestingness", "banality_resistance"]]
    .mean()
    .round(2)
    .reset_index()
)

avg_scores_df

## 12) Best mode counts

In [ ]:
winner_counts = judge_df[["task", "best_mode"]].drop_duplicates().groupby("best_mode").size().reset_index(name="wins")
winner_counts

## 13) Show one divergence vs full pipeline example

In [ ]:
task_result = all_results[0]

print("TASK:")
print(task_result["task"])
print("\n=== DIVERGENCE ONLY ===")
print(json.dumps(task_result["divergence_only"]["ideas"], ensure_ascii=False, indent=2))
print("\n=== FULL PIPELINE ===")
print(json.dumps(task_result["full_pipeline"]["ideas"], ensure_ascii=False, indent=2))